Cell 1: เชื่อมต่อ Google Drive และแตกไฟล์ Zip ลง Colab

In [ ]:
from google.colab import drive
import zipfile
import os
import glob

# 1. เชื่อมต่อ Google Drive
drive.mount('/content/drive')

# 2. ค้นหาไฟล์ .zip ใน Google Drive อัตโนมัติ
zip_files = glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
print("📌 พบไฟล์ Zip ใน Google Drive:")
for i, z in enumerate(zip_files):
    print(f"[{i}] {z}")

# เลือกไฟล์แรก (task4_image_classification.zip)
ZIP_PATH = zip_files[0]
print(f"\n📂 กำลังแตกไฟล์จาก: {ZIP_PATH}")

EXTRACT_DIR = '/content/data'
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

# 3. ค้นหา DATA_DIR ที่เก็บ train.csv
train_csv_paths = glob.glob('/content/data/**/train.csv', recursive=True)
DATA_DIR = os.path.dirname(train_csv_paths[0])
print(f"\n✅ พร้อมใช้งาน! DATA_DIR = '{DATA_DIR}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📌 พบไฟล์ Zip ใน Google Drive:
[0] /content/drive/MyDrive/CPE342/task4/task4_image_classification.zip

📂 กำลังแตกไฟล์จาก: /content/drive/MyDrive/CPE342/task4/task4_image_classification.zip

✅ พร้อมใช้งาน! DATA_DIR = '/content/data'


Cell 2: เตรียมไลบรารี, Dataset และการแบ่งข้อมูล Stratified 5-Fold

In [ ]:
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test')

SAVE_DIR = '/content/drive/MyDrive/CPE342_Task4_V2'
os.makedirs(SAVE_DIR, exist_ok=True)

CFG = {
    'seed': 42,
    'n_folds': 5,
    'train_folds': [0, 1, 2],   # รัน 3 Folds ก่อน (~35 นาทีบน T4) เพื่อเช็กคะแนน ถ้าเวลาเหลือค่อยเปลี่ยนเป็น [0, 1, 2, 3, 4]
    'img_size': (224, 224),     # เพิ่มความละเอียดจาก (128, 256) เป็น (224, 224) จับรายละเอียด UI ชัดขึ้น
    'batch_size': 32,
    'epochs': 8,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'num_classes': 5,
    'num_workers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
print("Device:", CFG['device'])

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_df.loc[val_idx, 'fold'] = fold

# Augmentation ที่เหมาะสมกับภาพหน้าจอเกม
train_transform = T.Compose([
    T.Resize(CFG['img_size']),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.RandomAffine(degrees=6, translate=(0.04, 0.04), scale=(0.92, 1.06)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.2, scale=(0.02, 0.12))
])

val_transform = T.Compose([
    T.Resize(CFG['img_size']),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class GameDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df.loc[idx, 'file_name']
        img_path = os.path.join(self.img_dir, file_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        if self.is_test:
            return image, self.df.loc[idx, 'id']
        else:
            label = torch.tensor(self.df.loc[idx, 'label'], dtype=torch.long)
            return image, label

print("✅ เตรียม Dataset เสร็จสมบูรณ์!")

Device: cuda
✅ เตรียม Dataset เสร็จสมบูรณ์!


Cell 3: ฟังก์ชันสำหรับเทรนและทำนายผล (รองรับการสลับเปลี่ยนโมเดลทำ Ensemble)

In [ ]:
def run_training_and_inference(model_name, lr=1e-4):
    print(f"\n🚀 เริ่มเทรนโมเดล: {model_name}")
    oof_probs = np.zeros((len(train_df), CFG['num_classes']))
    test_probs = np.zeros((len(test_df), CFG['num_classes']))

    test_dataset = GameDataset(test_df, TEST_IMG_DIR, val_transform, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=CFG['batch_size']*2, shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

    for fold in CFG['train_folds']:
        print(f"\n--- Fold {fold} ---")
        trn_df = train_df[train_df['fold'] != fold].reset_index(drop=True)
        val_df = train_df[train_df['fold'] == fold].reset_index(drop=True)
        val_indices = train_df[train_df['fold'] == fold].index.values

        # ปรับ Class Weight กำลัง 0.4 ช่วยบาลานซ์ Macro F1 ได้ดีกว่า 0.5
        class_weights = compute_class_weight('balanced', classes=np.unique(trn_df['label']), y=trn_df['label'])
        weights_tensor = torch.tensor(np.power(class_weights, 0.4), dtype=torch.float).to(CFG['device'])

        train_loader = DataLoader(GameDataset(trn_df, TRAIN_IMG_DIR, train_transform), batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
        val_loader = DataLoader(GameDataset(val_df, TRAIN_IMG_DIR, val_transform), batch_size=CFG['batch_size']*2, shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

        model = timm.create_model(model_name, pretrained=True, num_classes=CFG['num_classes'])
        model.to(CFG['device'])

        criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.05)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'] * len(train_loader), eta_min=1e-6)
        scaler = torch.amp.GradScaler('cuda')

        best_f1 = 0.0
        best_weight_path = os.path.join(SAVE_DIR, f"{model_name}_fold{fold}.pth")

        for epoch in range(CFG['epochs']):
            model.train()
            train_loss = 0.0
            for images, labels in tqdm(train_loader, desc=f"Ep {epoch+1}/{CFG['epochs']}", leave=False):
                images, labels = images.to(CFG['device']), labels.to(CFG['device'])
                optimizer.zero_grad()
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                train_loss += loss.item() * images.size(0)

            model.eval()
            val_preds, val_targets, val_probs = [], [], []
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(CFG['device']), labels.to(CFG['device'])
                    with torch.amp.autocast('cuda'):
                        outputs = model(images)
                        probs = torch.softmax(outputs, dim=1)
                    val_probs.append(probs.cpu().numpy())
                    val_preds.append(probs.argmax(dim=1).cpu().numpy())
                    val_targets.append(labels.cpu().numpy())

            val_probs = np.concatenate(val_probs, axis=0)
            val_preds = np.concatenate(val_preds, axis=0)
            val_targets = np.concatenate(val_targets, axis=0)
            score = f1_score(val_targets, val_preds, average='macro')
            print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss/len(trn_df):.4f} | Val Macro F1: {score:.5f}")

            if score > best_f1:
                best_f1 = score
                oof_probs[val_indices] = val_probs
                torch.save(model.state_dict(), best_weight_path)

        print(f"⭐ Fold {fold} Best Macro F1: {best_f1:.5f}")

        # ทำนาย Test Set ด้วย TTA (ภาพปกติ + ภาพ Flip ซ้ายขวา)
        model.load_state_dict(torch.load(best_weight_path))
        model.eval()
        fold_test_probs = []
        with torch.no_grad():
            for images, _ in tqdm(test_loader, desc=f"Predict Fold {fold}", leave=False):
                images = images.to(CFG['device'])
                with torch.amp.autocast('cuda'):
                    p1 = torch.softmax(model(images), dim=1)
                    p2 = torch.softmax(model(torch.flip(images, dims=[3])), dim=1)
                    probs = (p1 + p2) / 2.0
                fold_test_probs.append(probs.cpu().numpy())
        test_probs += np.concatenate(fold_test_probs, axis=0) / len(CFG['train_folds'])

    np.save(os.path.join(SAVE_DIR, f"{model_name}_oof.npy"), oof_probs)
    np.save(os.path.join(SAVE_DIR, f"{model_name}_test.npy"), test_probs)
    return oof_probs, test_probs

print("✅ โหลดฟังก์ชันเทรนเรียบร้อย!")

✅ โหลดฟังก์ชันเทรนเรียบร้อย!


Cell 4: รันโมเดลตัวที่ 1 (convnext_base.fb_in22k_ft_in1k)

In [ ]:
oof_convnext, test_convnext = run_training_and_inference(
    model_name='convnext_base.fb_in22k_ft_in1k',
    lr=1e-4
)

# เซฟผลลัพธ์ของโมเดลตัวที่ 1 เผื่อไว้ส่งเช็กคะแนนทันที
sub_m1 = pd.DataFrame({'id': test_df['id'], 'task4': test_convnext.argmax(axis=1)})
sub_m1.to_csv(os.path.join(SAVE_DIR, 'submission_task4_convnext_base.csv'), index=False)
print("✅ เซฟ submission_task4_convnext_base.csv ลง Google Drive แล้ว!")


🚀 เริ่มเทรนโมเดล: convnext_base.fb_in22k_ft_in1k

--- Fold 0 ---


model.safetensors: reconstructing file:   0%|          |  0.00B /  354MB            

model.safetensors: downloading bytes:           |  0.00B            

Ep 1/8:   0%|          | 0/788 [00:00<?, ?it/s]

/tmp/ipykernel_502/77196744.py:45: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 01 | Train Loss: 0.4272 | Val Macro F1: 0.97531


... (64 outputs left out)

Ep 8/8:   0%|          | 0/788 [00:00<?, ?it/s]

Epoch 08 | Train Loss: 0.2447 | Val Macro F1: 0.98650
⭐ Fold 2 Best Macro F1: 0.98650


Predict Fold 2:   0%|          | 0/405 [00:00<?, ?it/s]

✅ เซฟ submission_task4_convnext_base.csv ลง Google Drive แล้ว!


In [ ]:
import pandas as pd

sub = pd.read_csv('submission_task4_convnext_base.csv')
print("จำนวนแถวทั้งหมด:", len(sub))
print("\nสัดส่วนคลาสที่ทำนายได้ (%):")
print((sub['task4'].value_counts(normalize=True).sort_index() * 100).round(2))

จำนวนแถวทั้งหมด: 25889

สัดส่วนคลาสที่ทำนายได้ (%):
task4
0    18.61
1    23.68
2    14.81
3    26.41
4    16.49
Name: proportion, dtype: float64
